# 🏪 Retail Cost Control Analysis
### Are our stores staying within budget?

In this notebook we work with a real-world inspired dataset from a retail chain.  
Each store has **10 types of recurring costs** — cleaning, lunch catering, celebrations, electricity, and more.

**Our goal:** identify which stores are exceeding cost caps so the operations team knows where to act.

---
**Dataset files:**
- `stores.csv` — one row per store (size in m², number of employees, city)
- `cost_types.csv` — the 10 cost categories and how they should be normalized
- `store_costs.csv` — monthly cost amounts per store per cost type (12 months, format YYYY-MM)


---
## Part 1 — Loading & Exploring the Data


In [1]:
import pandas as pd
import numpy as np

stores      = pd.read_csv("stores.csv")
cost_types  = pd.read_csv("cost_types.csv")
store_costs = pd.read_csv("store_costs.csv")


#### 👀 Take a look at each table

In [2]:
stores.head()

,store_id,store_name,city,size_m2,num_employees
0,1,Berlin Mitte,Berlin,1200,42
1,2,Hamburg Altona,Hamburg,950,33
2,3,Munich Schwabing,Munich,1450,51
3,4,Frankfurt Sachsenhausen,Frankfurt,800,28
4,5,Cologne Ehrenfeld,Cologne,1100,38


In [3]:
cost_types.head(10)

,cost_id,cost_name,normalization_method,description
0,1,Cleaning,per_m2,Daily and deep cleaning services
1,2,Lunch Catering,per_employee,Daily lunch provided to staff
2,3,Monthly Celebrations,per_employee,"Birthday, holiday and team celebration costs"
3,4,Office Supplies,per_employee,"Pens, paper, printer ink and general supplies"
4,5,Security,per_m2,Security personnel and systems
5,6,Maintenance,per_m2,Building repairs and upkeep
6,7,Electricity,per_m2,Monthly electricity consumption
7,8,Internet & Phones,per_employee,"Phone lines, mobile contracts and internet"
8,9,First Aid & Safety,per_employee,"First aid kits, fire extinguisher checks, safe..."
9,10,Waste Disposal,per_m2,Trash collection and recycling services


In [4]:
store_costs.head()

,store_id,cost_id,month,raw_amount
0,1,1,2024-01,7205.38
1,1,1,2024-02,6853.34
2,1,1,2024-03,7289.07
3,1,1,2024-04,7774.37
4,1,1,2024-05,6800.18


---
#### 🔍 Exploring `store_costs` — dimensions & date range


In [ ]:
# How many rows does store_costs have?
# Hint: df.shape
n_rows = store_costs.
print(n_rows)


In [5]:
# How many unique stores are there?
# Hint: df["col"].nunique()
n_stores = store_costs["store_id"].nunique()
print(n_stores)


10


In [ ]:
# How many unique cost types are there?
# Hint: df["col"].nunique()
n_costs = store_costs["_____"]._____
print(n_costs)


In [ ]:
# What is the earliest month in the dataset?
# Hint: df["col"].min()
min_month = store_costs["month"]._____
print(min_month)


In [ ]:
# What is the latest month in the dataset?
# Hint: df["col"].max()
max_month = store_costs["month"]._____
print(max_month)


In [ ]:
# Are there any missing values?
# Hint: df.isnull().sum()
nulls = store_costs._____._____
print(nulls)


---
#### 🔍 Exploring `stores` — sizes & headcount


In [ ]:
# Which store has the most employees?
# Hint: df.sort_values("col", ascending=False).head(1)
biggest_team = stores.sort_values("_____", ascending=False)._____
print(biggest_team)


In [ ]:
# Which store has the smallest size in m²?
# Hint: df.sort_values("col").head(1)
smallest_store = stores.sort_values("_____")._____
print(smallest_store)


In [ ]:
# What is the average store size in m²?
# Hint: df["col"].mean()
avg_size = stores["_____"]._____
print(round(avg_size, 1))


In [ ]:
# What is the average number of employees per store?
# Hint: df["col"].mean()
avg_employees = stores["_____"]._____
print(round(avg_employees, 1))


---
#### 🔍 Exploring `cost_types` — normalization methods


In [ ]:
# What are the unique normalization methods?
# Hint: df["col"].unique()
norm_methods = cost_types["normalization_method"]._____
print(norm_methods)


In [ ]:
# How many cost types use each normalization method?
# Hint: df["col"].value_counts()
norm_counts = cost_types["_____"]._____
print(norm_counts)


In [ ]:
# Which cost types are normalized per_m2?
# Hint: df[df["col"] == "value"][["col1", "col2"]]
per_m2 = cost_types[cost_types["_____"] == "per_m2"][["cost_name", "normalization_method"]]
print(per_m2)


In [ ]:
# Which cost types are normalized per_employee?
# Hint: df[df["col"] == "value"][["col1", "col2"]]
per_emp = cost_types[cost_types["_____"] == "_____"][["cost_name", "normalization_method"]]
print(per_emp)


---
## Part 2 — Joining the Tables

Right now `store_costs` only has IDs — we need to enrich it with store details and cost descriptions.  
We'll do two merges to build one flat table.


In [ ]:
# Step 1 — join store_costs with stores
# Hint: left_df.merge(right_df, on="shared_col")
df = store_costs.merge(stores, on="store_id")

df.head()


In [ ]:
# Step 2 — join the result with cost_types
# Hint: left_df.merge(right_df, on="shared_col")
df = df.merge(_____, on="cost_id")

print(df.shape)
df.head()


In [ ]:
# Convert month to a Period — proper date type with month precision
# no fake day needed, sorts correctly, and supports .dt.month / .dt.year

# Step 1 — convert to datetime first
# Hint: pd.to_datetime(df["col"])
df["month"] = pd.to_datetime(df["_____"])

# Step 2 — convert to Period with monthly frequency
# Hint: series.dt.to_period("freq") — use "M" for monthly
df["month"] = df["month"].dt.to_period("_____")

# Check — what does it look like now?
print(df["month"].dtype)
print(df["month"].head())


---
## Part 3 — Normalizing Costs

Here is the core business problem:

> **Munich Schwabing** has 51 employees and 1450 m².  
> **Leipzig Zentrum** has 26 employees and 750 m².  
>
> If Munich spends €5,000 on lunch and Leipzig spends €2,500 — who is really overspending?

We can't compare raw amounts fairly. We need to **normalize** by the right unit.

- Costs like *Lunch* or *Celebrations* → divide by `num_employees`  
- Costs like *Cleaning* or *Electricity* → divide by `size_m2`  

The `normalization_method` column tells us which one to use.


In [ ]:
# Hint: np.where(condition, value_if_true, value_if_false)
# condition  → df["normalization_method"] == "per_employee"
# if true    → df["raw_amount"] / df["num_employees"]
# if false   → df["raw_amount"] / df["size_m2"]

df["normalized_cost"] = np.where(
    df["normalization_method"] == "per_employee",
    df["raw_amount"] / df["_____"],
    df["raw_amount"] / df["_____"]
)

df[["store_name", "cost_name", "normalization_method", "raw_amount", "normalized_cost"]].head(10)


#### 🤔 Sanity check — does normalization change the picture?

Filter only **Lunch Catering** for one month and compare stores.


In [ ]:
# Step 1 — filter the data
# Hint: df[(df["col1"] == val1) & (df["col2"] == val2)]
lunch = df[(df["_____"] == "Lunch Catering") & (df["_____"] == "2024-06")]

# Step 2 — select columns and sort
# Hint: df[["col1","col2"]].sort_values("col", ascending=False)
lunch_sorted = lunch[["store_name", "num_employees", "raw_amount", "normalized_cost"]].sort_values("_____", ascending=False)

print(lunch_sorted)


---
## Part 4 — Benchmarks & Caps

Now that costs are normalized, we define a **fair benchmark** per cost type.

**Strategy:**
1. Compute the **median** normalized cost per cost type (across all stores and months)
2. Set the **cap** at benchmark × 1.2 — stores can be up to 20% above median before being flagged
3. Flag rows where `normalized_cost > cap`

> We use the median instead of the mean because it's more resistant to outliers —  
> exactly what we're trying to detect!


In [ ]:
# Step 1 — compute median normalized cost per cost type
# Hint: df.groupby("col")["target_col"].median().reset_index()
benchmark = df.groupby("_____")["normalized_cost"]._____().reset_index()

# Step 2 — rename the column
# Hint: df.rename(columns={"old_name": "new_name"})
benchmark = benchmark.rename(columns={"normalized_cost": "benchmark"})

print(benchmark)


In [ ]:
# Step 1 — merge benchmark back into df
# Hint: left_df.merge(right_df, on="shared_col")
df = df.merge(_____, on="_____")

# Step 2 — compute the cap: 20% above benchmark
# Hint: df["new_col"] = df["existing_col"] * number
df["cap"] = df["_____"] * 1.5

df[["store_name", "cost_name", "normalized_cost", "benchmark", "cap"]].head(10)


In [ ]:
# Step 1 — create the boolean flag
# Hint: df["new_col"] = df["col_a"] > df["col_b"]
df["over_cap"] = df["cap"] > df["benchmark"]

true /false - true = 1

In [ ]:
# Step 2 — count violations
# Hint: boolean_series.sum() counts the True values
total_rows       = len(____)
total_violations = df["over_cap"].sum



In [ ]:
# Step 3 - Violation rate
# Hint: violations / total * 100
violation_rate = round(_____ / _____ * 100, 1)



In [ ]:
print("Total rows:     ", total_rows)
print("Violations:     ", total_violations)
print("Violation rate: ", violation_rate, "%")


---
## Part 5 — Who Needs Our Attention?


#### 🏪 Which stores have the most violations?

In [ ]:
# Step 1 — count violations per store
# Hint: df.groupby("col")["bool_col"].sum().reset_index()
store_violations = df.groupby("_____")["over_cap"]._____.reset_index()

# Step 2 — sort
# Hint: df.sort_values("col", ascending=False)
store_violations_sorted = store_violations.sort_values("_____", ascending=False)

print(store_violations_sorted)


In [ ]:
# Violations per store per month — is it a recurring problem or a one-off?
# Hint: df.groupby(["col1", "col2"])["bool_col"].sum().reset_index()
store_month_violations = df.groupby(["_____", "_____"])["over_cap"]._____.reset_index()

print(store_month_violations)

#### 💸 Which cost types are most frequently exceeded?

In [ ]:
# Step 1 — count violations per cost type
# Hint: df.groupby("col")["bool_col"].sum().reset_index()
cost_violations = df.groupby("_____")["over_cap"]._____.reset_index()

# Step 2 — sort
# Hint: df.sort_values("col", ascending=False)
cost_violations_sorted = cost_violations.sort_values("_____", ascending=False)

print(cost_violations_sorted)


> 💡 **Think about it:** if a cost type is over-cap in 7 out of 10 stores,  
> maybe the **cap is too tight** — not all stores are misbehaving.  
> This is a business judgment, not just a code exercise.


---
## Part 6 — Violations Over Time

A total count tells us *who* is over cap — but not *when* or *whether it's getting worse*.

A store with 12 violations spread across 12 months has a **structural problem**.  
A store with 3 violations all in December probably just overspent on celebrations.

The `month` column is already in `YYYY-MM` format — ready to use as-is.


In [ ]:
# How many violations happened each month across all stores?
# Hint: df.groupby("col")["bool_col"].sum().reset_index()
monthly_violations = df.groupby("_____")["over_cap"]._____.reset_index()

print(monthly_violations)


#### Zoom in on one store — which months and which costs?

Change the store name to investigate any store you like.


In [ ]:
# Step 1 — filter to one store
# Hint: df[df["col"] == "value"]
store_name = "Hamburg Altona"
store_df   = df[df["_____"] == _____]

# Step 2 — count violations per month and cost type
# Hint: df.groupby(["col1", "col2"])["bool_col"].sum().reset_index()
store_timeline = store_df.groupby(["_____", "_____"])["over_cap"]._____.reset_index()

# Step 3 — filter to only months with at least one violation
# Hint: df[df["col"] > value]
store_timeline_filtered = store_timeline[store_timeline["_____"] > 0]

print(store_timeline_filtered)


---
## ✅ Summary

| Skill | Where |
|---|---|
| Loading and exploring DataFrames | Part 1 |
| `.min()`, `.max()`, `.mean()`, `.value_counts()` | Part 1 |
| Filtering rows | Part 1 |
| Merging multiple tables | Part 2 |
| Conditional column with `np.where` | Part 3 |
| Filtering with multiple conditions | Part 3 |
| `groupby` + `median` for benchmarking | Part 4 |
| Merging aggregated results back | Part 4 |
| Boolean flagging | Part 4 |
| `groupby` + `sum` on boolean column | Part 5 |
| Time-based analysis | Part 6 |
| Exporting for Power BI | Part 7 |
| Bar charts and line charts with matplotlib | Part 8 |

**Key insight:** normalizing costs by m² or by headcount lets you compare stores of very different sizes fairly.  
A store twice the size *should* spend roughly twice as much on cleaning — the cap accounts for that automatically.
